# nb06 — RetinaNet R1: Data-Grounded Anchors

**Motivation.** nb05 (R0) trained RetinaNet with **COCO default anchors** (32, 64, 128, 256, 512) and reached mAP=0.0410 on the full val set. EDA (nb01) showed that VisDrone's object-size distribution differs sharply from COCO — the k-means anchors span **23.8×33.7 px to 336.4×221.7 px** in original image space. R0's default anchors are poorly matched to the dominant small/medium objects, so most anchors never overlap a real object at IoU > 0.5 → low AP_small (0.0148).

**R1 = R0 + k-means anchors.** We convert the nb01 k-means anchors to letterbox-1280 coordinates and rebuild RetinaNet with them. Everything else (protocol, optimizer, 1 epoch, clip split) is identical.

**Warm-start.** The backbone and FPN are loaded from `retinanet_R0.pth`. Classification and regression heads are **re-initialized** because the number of anchors per location changed from 9 → 9 (same count) but the actual anchor shape changes, which means the regression targets shift. Re-initializing the heads is standard practice — the backbone still carries useful features.

**Artifacts produced:**
- `retinanet_R1.pth` — checkpoint
- `val_preds_retinanet_R1.json` — full-val predictions
- `metrics_retinanet_R1.json`, `speed_retinanet_R1.json`, `train_log_retinanet_R1.json`
- `submission_retinanet_R1.csv` — Kaggle submission
- W&B run `retinanet_R1`

In [1]:
import sys, json, time, shutil, os
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision.ops import box_iou
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models.detection.retinanet import RetinaNetClassificationHead, RetinaNetRegressionHead
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.transforms import functional as TF

# ---- Paths -----------------------------------------------------------------
COMP_DIR       = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
DATASET_DIR    = Path("/kaggle/input/datasets/suchok/my-dataset")
OUTPUTS_DIR    = Path("/kaggle/input/datasets/suchok/outputs")
RETINANET1_DIR = Path("/kaggle/input/datasets/suchok/retinanet-1")
WORK_DIR       = Path("/kaggle/working")

assert COMP_DIR.exists(),       f"missing competition: {COMP_DIR}"
assert DATASET_DIR.exists(),    f"missing my-dataset: {DATASET_DIR}"
assert OUTPUTS_DIR.exists(),    f"missing outputs: {OUTPUTS_DIR}"
assert RETINANET1_DIR.exists(), f"missing retinanet-1: {RETINANET1_DIR}"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ---- common.py --------------------------------------------------------------
sys.path.insert(0, str(DATASET_DIR))
import common, importlib
importlib.reload(common)
from common import (
    set_seeds, SEED, CLASS_NAMES, NUM_CLASSES,
    load_splits, index_split, filter_by_clips,
    VisDroneDetectionDataset, detection_collate,
    build_fullval_gt, coco_eval_summary,
    save_checkpoint, save_metrics,
    IMAGENET_MEAN, IMAGENET_STD,
)

set_seeds(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| torch:", torch.__version__)

# ---- Speed flags (safe; make training faster) -------------------------------
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

# ---- Locked protocol --------------------------------------------------------
chosen = json.load(open(OUTPUTS_DIR / "chosen_preprocessing.json"))
STRAT  = chosen["strategy"]
SIZE   = int(chosen["size"])
print("Locked preprocessing:", STRAT, SIZE)

train_clips, val_clips = load_splits(OUTPUTS_DIR / "splits")
train_index = filter_by_clips(index_split(COMP_DIR / "train"), train_clips)
val_index   = filter_by_clips(index_split(COMP_DIR / "val"),   val_clips)
print(f"train frames: {len(train_index):,} | val frames: {len(val_index):,}")

TEST_CSV = COMP_DIR / "test_images.csv"

# ---- Restore R0 checkpoint + GT --------------------------------------------
for name in ["retinanet_R0.pth", "gt_fullval_ignore.json"]:
    src = RETINANET1_DIR / name
    dst = WORK_DIR / name
    if src.exists():
        shutil.copy(src, dst)
        print(f"restored: {name}")
    else:
        raise FileNotFoundError(f"missing in retinanet-1: {name}")

R0_CKPT          = WORK_DIR / "retinanet_R0.pth"
gt_fullval_ignore = WORK_DIR / "gt_fullval_ignore.json"
assert R0_CKPT.exists()
assert gt_fullval_ignore.exists()

DEVICE: cuda | torch: 2.11.0+cu130
Locked preprocessing: letterbox 1280
train frames: 24,201 | val frames: 2,846
restored: retinanet_R0.pth
restored: gt_fullval_ignore.json


In [2]:
# Look for the anchor stats file in multiple candidate locations.
anchor_candidates = [
    OUTPUTS_DIR / "anchor_stats.json",
    OUTPUTS_DIR / "kmeans_anchors.json",
    OUTPUTS_DIR / "anchors.json",
    Path("/kaggle/input/datasets/suchok/outputs-3") / "anchor_stats.json",
]
anchor_file = next((p for p in anchor_candidates if p.exists()), None)

if anchor_file is None:
    print("⚠️  no anchor stats file found. Falling back to nb01-reported values.")
    # Values from draft_to_report.md §1.2
    kmeans_anchors_orig = [
        (23.8, 33.7), (37.5, 52.1), (58.3, 71.4),
        (81.2, 94.5), (110.5, 128.0), (149.3, 173.2),
        (199.4, 205.7), (260.1, 231.3), (336.4, 221.7),
    ]
else:
    print(f"loaded anchor stats from: {anchor_file}")
    with open(anchor_file) as f:
        stats = json.load(f)
    # Try common keys
    if "kmeans_anchors" in stats:
        raw = stats["kmeans_anchors"]
    elif "anchors" in stats:
        raw = stats["anchors"]
    else:
        raise KeyError(f"unrecognised anchor stats schema: {list(stats.keys())}")
    kmeans_anchors_orig = [tuple(a) for a in raw]

print(f"k-means anchors (original image px): {len(kmeans_anchors_orig)}")
for i, (w, h) in enumerate(kmeans_anchors_orig):
    print(f"  anchor {i}: w={w:.1f}, h={h:.1f}")

# ---- Convert to letterbox-1280 space ----
# Letterbox scale depends on the source image size. VisDrone images are
# mostly 3840x2160 (4K) or 1920x1080 (HD). The letterbox scale for a
# 3840x2160 image to fit in 1280x1280 is scale = 1280 / 3840 = 0.333.
# For 1920x1080 it's 1280 / 1920 = 0.667. To make anchors work across both
# resolutions, we compute the *geometric mean* of the two dominant scales.
# This is an approximation; a proper solution would use FPN-level assignment.
scales_4k = 1280 / 3840     # 0.333
scales_hd = 1280 / 1920     # 0.667
scale_gmean = float(np.sqrt(scales_4k * scales_hd))   # ~0.471
print(f"\nletterbox scale (geometric mean): {scale_gmean:.4f}")

anchors_lb1280 = [(w * scale_gmean, h * scale_gmean) for w, h in kmeans_anchors_orig]
print("k-means anchors (letterbox-1280):")
for i, (w, h) in enumerate(anchors_lb1280):
    print(f"  anchor {i}: w={w:.1f}, h={h:.1f}")

⚠️  no anchor stats file found. Falling back to nb01-reported values.
k-means anchors (original image px): 9
  anchor 0: w=23.8, h=33.7
  anchor 1: w=37.5, h=52.1
  anchor 2: w=58.3, h=71.4
  anchor 3: w=81.2, h=94.5
  anchor 4: w=110.5, h=128.0
  anchor 5: w=149.3, h=173.2
  anchor 6: w=199.4, h=205.7
  anchor 7: w=260.1, h=231.3
  anchor 8: w=336.4, h=221.7

letterbox scale (geometric mean): 0.4714
k-means anchors (letterbox-1280):
  anchor 0: w=11.2, h=15.9
  anchor 1: w=17.7, h=24.6
  anchor 2: w=27.5, h=33.7
  anchor 3: w=38.3, h=44.5
  anchor 4: w=52.1, h=60.3
  anchor 5: w=70.4, h=81.6
  anchor 6: w=94.0, h=97.0
  anchor 7: w=122.6, h=109.0
  anchor 8: w=158.6, h=104.5


## Anchor configuration for RetinaNet

RetinaNet's `AnchorGenerator` expects **one size per FPN level** (5 levels: P3–P7) and **one aspect ratio tuple per level**. We have 9 k-means anchors → we **sort them by area** and use the 9 values as follows:

- Use **3 anchor sizes per FPN level** (grouping the 9 k-means anchors into 3 groups of 3 by area).
- Set **aspect ratios** to `(0.5, 1.0, 2.0)` for each size (3 sizes × 3 ratios = 9 anchors per location, same total as COCO).

This gives 9 anchors per location, spread across the object-size range observed in VisDrone.

**Why geometric grouping, not just 9 sizes at once?** torchvision's `AnchorGenerator` supports `sizes=((a,b,c), (a,b,c), ...)` for multiple sizes per level. Using 3 per level is the standard choice (matches COCO default structure) and gives FPN multi-scale handling — small anchors land at P3/P4, large at P6/P7.

In [3]:
def build_retinanet_r1(anchors_lb1280, score_thresh=0.05, nms_thresh=0.5):
    """RetinaNet with anchors derived from VisDrone k-means (letterbox-1280 space).

    anchors_lb1280: list of (w, h) tuples in letterbox pixels; length 9.
    Sorts by area, groups into 3 sizes per FPN level, 3 levels + 2 repeats.
    """
    # Sort by area ascending
    sorted_anchors = sorted(anchors_lb1280, key=lambda wh: wh[0] * wh[1])
    # Group into 3 groups of 3
    sizes_grouped = [
        tuple(round(w) for w, h in sorted_anchors[0:3]),     # small
        tuple(round(w) for w, h in sorted_anchors[3:6]),     # medium
        tuple(round(w) for w, h in sorted_anchors[6:9]),     # large
    ]
    # FPN has 5 levels — repeat the pattern (small, medium, large, large, large)
    # This gives P6/P7 the largest anchors (appropriate for large objects).
    fpn_sizes = (
        sizes_grouped[0],
        sizes_grouped[0],
        sizes_grouped[1],
        sizes_grouped[2],
        sizes_grouped[2],
    )
    aspect_ratios = ((0.5, 1.0, 2.0),) * 5

    m = retinanet_resnet50_fpn_v2(
        weights=None, weights_backbone=None,
        min_size=SIZE, max_size=SIZE,
        score_thresh=score_thresh, nms_thresh=nms_thresh,
        detections_per_img=500,
    )
    m.anchor_generator = AnchorGenerator(sizes=fpn_sizes, aspect_ratios=aspect_ratios)
    num_anchors = m.anchor_generator.num_anchors_per_location()[0]

    in_channels = m.head.classification_head.cls_logits.in_channels
    m.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels, num_anchors=num_anchors,
        num_classes=NUM_CLASSES + 1,
        norm_layer=lambda num_channels: torch.nn.GroupNorm(32, num_channels),
    )
    in_channels_reg = m.head.regression_head.bbox_reg.in_channels
    m.head.regression_head = RetinaNetRegressionHead(
        in_channels=in_channels_reg, num_anchors=num_anchors,
        norm_layer=lambda num_channels: torch.nn.GroupNorm(32, num_channels),
    )
    print(f"R1 anchor sizes per FPN level: {fpn_sizes}")
    print(f"anchors per location: {num_anchors}")
    return m


model_r1 = build_retinanet_r1(anchors_lb1280).to(DEVICE)

# ---- Warm-start: load R0 state dict, but ONLY the layers whose shapes match ----
r0_state = torch.load(R0_CKPT, map_location=DEVICE)
r1_state = model_r1.state_dict()
loaded, skipped = [], []
for k, v in r0_state.items():
    if k in r1_state and v.shape == r1_state[k].shape:
        r1_state[k] = v
        loaded.append(k)
    else:
        skipped.append(k)
model_r1.load_state_dict(r1_state, strict=True)
print(f"warm-start: loaded {len(loaded)} tensors, skipped {len(skipped)} (heads re-init)")

n_params = sum(p.numel() for p in model_r1.parameters() if p.requires_grad)
print(f"trainable params: {n_params/1e6:.2f}M")

R1 anchor sizes per FPN level: ((11, 18, 27), (11, 18, 27), (38, 52, 70), (94, 123, 159), (94, 123, 159))
anchors per location: 9
warm-start: loaded 358 tensors, skipped 4 (heads re-init)
trainable params: 36.50M


In [4]:
from torch.utils.data import DataLoader

train_ds = VisDroneDetectionDataset(train_index, STRAT, SIZE, is_train=False)
train_loader = DataLoader(
    train_ds, batch_size=2, shuffle=True, num_workers=2,
    collate_fn=detection_collate, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    VisDroneDetectionDataset(val_index, STRAT, SIZE, is_train=False),
    batch_size=2, shuffle=False, num_workers=2,
    collate_fn=detection_collate, pin_memory=True,
)
print(f"train batches: {len(train_loader):,} | val batches: {len(val_loader):,}")

# Optimizer / scheduler — identical to nb03/04/05
params = [p for p in model_r1.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=5e-3, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Smoke test — one batch, check losses are finite
model_r1.train()
imgs, tgts = next(iter(train_loader))
imgs = [i.to(DEVICE) for i in imgs]
tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
with torch.no_grad():
    loss_dict = model_r1(imgs, tgts)
print("smoke loss:", {k: round(float(v.detach()), 4) for k, v in loss_dict.items()})

train batches: 12,100 | val batches: 1,423
smoke loss: {'classification': 1.1482, 'bbox_regression': 0.65}


In [5]:
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

import wandb
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)


def train_one_epoch_with_progress(model, loader, optimizer, scheduler,
                                  device, epoch, wandb_run=None,
                                  log_every=100, wandb_log_every=20):
    """Training loop with step prints AND W&B logging.

    Prints every `log_every` batches so you can monitor progress in the log
    even when tqdm is not captured by papermill.
    """
    model.train()
    running = {"loss": 0.0, "loss_cls": 0.0, "loss_box": 0.0}
    n_batches = 0
    t_start = time.time()

    for step, (images, targets) in enumerate(tqdm(loader, desc=f"R1 epoch {epoch}")):
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()}
                   for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        losses.backward()
        optimizer.step()

        running["loss"]     += float(losses.detach())
        running["loss_cls"] += float(loss_dict.get("classification", 0).detach())
        running["loss_box"] += float(loss_dict.get("bbox_regression", 0).detach())
        n_batches += 1

        # ---- Step print every `log_every` batches ----
        if step % log_every == 0 and step > 0:
            elapsed = time.time() - t_start
            s_per_batch = elapsed / (step + 1)
            remaining = (len(loader) - step - 1) * s_per_batch
            print(f"[{step}/{len(loader)}] "
                  f"elapsed={elapsed/60:.1f}min "
                  f"{s_per_batch:.2f}s/batch "
                  f"ETA={remaining/3600:.2f}h "
                  f"loss={float(losses.detach()):.4f}",
                  flush=True)

        # ---- W&B log every `wandb_log_every` batches ----
        if step % wandb_log_every == 0 and wandb_run is not None:
            wandb_run.log({
                "train/loss":     float(losses.detach()),
                "train/loss_cls": float(loss_dict.get("classification", 0).detach()),
                "train/loss_box": float(loss_dict.get("bbox_regression", 0).detach()),
                "train/step":     step,
            })

    scheduler.step()
    mean = {k: v / max(1, n_batches) for k, v in running.items()}
    return mean, time.time() - t_start

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: suchok-pn (suchok-pn-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
run = wandb.init(
    project="visdrone-det",
    name="retinanet_R1",
    config={
        "model": "retinanet_resnet50_fpn_v2",
        "variant": "R1_kmeans_anchors",
        "strategy": STRAT, "size": SIZE,
        "epochs": 1, "lr": 5e-3, "batch_size": 2,
        "anchors_orig_px": kmeans_anchors_orig,
        "anchors_lb1280":  anchors_lb1280,
        "warmstart": "retinanet_R0.pth",
        "seed": SEED,
    },
    reinit=True,
)

t_start = time.time()
train_stats, train_seconds = train_one_epoch_with_progress(
    model_r1, train_loader, optimizer, scheduler, DEVICE,
    epoch=1, wandb_run=run, log_every=100, wandb_log_every=20,
)
print(f"\ntrain stats: {train_stats}")
print(f"train wall time: {train_seconds/3600:.2f} h")

# ---- Save checkpoint + train log ----
save_checkpoint(model_r1, WORK_DIR / "retinanet_R1.pth")
with open(WORK_DIR / "train_log_retinanet_R1.json", "w") as f:
    json.dump({
        "epoch": 1,
        "mean_loss":      train_stats["loss"],
        "mean_loss_cls":  train_stats["loss_cls"],
        "mean_loss_box":  train_stats["loss_box"],
        "wall_seconds":   train_seconds,
        "batch_size": 2, "lr": 5e-3,
        "anchors_lb1280": anchors_lb1280,
    }, f, indent=2)

wandb.log({"train/epoch_loss": train_stats["loss"],
           "train/epoch_time_h": train_seconds/3600})
print("checkpoint + train_log saved")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261107_135523-nu184o9s
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run retinanet_R1
wandb: ⭐️ View project at https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det
wandb: 🚀 View run at https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det/runs/nu184o9s


R1 epoch 1:   0%|          | 0/12100 [00:00<?, ?it/s]

[100/12100] elapsed=2.7min 1.62s/batch ETA=5.40h loss=1.2573
[200/12100] elapsed=5.1min 1.53s/batch ETA=5.07h loss=1.0064
[300/12100] elapsed=7.5min 1.50s/batch ETA=4.93h loss=0.9234
[400/12100] elapsed=10.0min 1.49s/batch ETA=4.84h loss=0.7987
[500/12100] elapsed=12.4min 1.48s/batch ETA=4.77h loss=0.8534
[600/12100] elapsed=14.8min 1.48s/batch ETA=4.71h loss=0.9266
[700/12100] elapsed=17.2min 1.47s/batch ETA=4.66h loss=0.9606
[800/12100] elapsed=19.6min 1.47s/batch ETA=4.61h loss=0.7118
[900/12100] elapsed=22.0min 1.47s/batch ETA=4.56h loss=0.9293
[1000/12100] elapsed=24.4min 1.46s/batch ETA=4.51h loss=0.8749
[1100/12100] elapsed=26.8min 1.46s/batch ETA=4.47h loss=0.6523
[1200/12100] elapsed=29.3min 1.46s/batch ETA=4.43h loss=0.8127
[1300/12100] elapsed=31.7min 1.46s/batch ETA=4.38h loss=0.7627
[1400/12100] elapsed=34.1min 1.46s/batch ETA=4.34h loss=0.8201
[1500/12100] elapsed=36.5min 1.46s/batch ETA=4.29h loss=0.5626
[1600/12100] elapsed=38.9min 1.46s/batch ETA=4.25h loss=0.8157
[170

In [7]:
@torch.no_grad()
def run_detector_eval(detector, image_rgb, score_thr=0.001, device=None):
    """Safe eval override — forces detector.eval() before forward."""
    if device is None:
        device = next(detector.parameters()).device
    detector.eval()
    tensor = TF.to_tensor(image_rgb)
    tensor = TF.normalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
    out = detector([tensor.to(device)])[0]
    keep = out["scores"] >= score_thr
    return (out["boxes"][keep].cpu().numpy(),
            out["labels"][keep].cpu().numpy(),
            out["scores"][keep].cpu().numpy())


def predict_with_strategy_eval(detector, image_path, strategy, size,
                               score_thr=0.001, device=None):
    from common import apply_strategy, boxes_to_original
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    h0, w0 = img.shape[:2]
    inp, meta = apply_strategy(img, strategy, size, seed_key=image_path)
    t0 = time.time()
    boxes, labels, scores = run_detector_eval(detector, inp,
                                              score_thr=score_thr, device=device)
    dt = time.time() - t0
    boxes_orig = boxes_to_original(boxes, meta, strategy, w0, h0)
    return boxes_orig, labels, scores, dt, meta


def predictions_to_coco_eval(df_index, detector, strategy, size, out_json,
                              score_thr=0.001, device=None):
    import contextlib
    preds = []
    id_map = {row["image_path"]: i for i, (_, row) in enumerate(
        df_index.iterrows(), start=1)}
    valid_cat_ids = set(range(1, NUM_CLASSES + 1))
    for _, row in tqdm(df_index.iterrows(), total=len(df_index),
                       desc=f"pred {strategy}-{size}"):
        boxes, labels, scores, _, _ = predict_with_strategy_eval(
            detector, row["image_path"], strategy, size,
            score_thr=score_thr, device=device)
        img_id = id_map[row["image_path"]]
        for (x1, y1, x2, y2), l, s in zip(boxes, labels, scores):
            if int(l) not in valid_cat_ids:
                continue
            preds.append({
                "image_id": int(img_id),
                "category_id": int(l),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "score": float(s),
            })
    with open(out_json, "w") as f:
        json.dump(preds, f)
    print(f"wrote {len(preds):,} predictions -> {out_json}")


def coco_eval_summary_safe(gt_json, dt_json):
    import contextlib, io
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO(gt_json)
        if os.path.getsize(dt_json) == 0 or json.load(open(dt_json)) == []:
            return {k: 0.0 for k in ["mAP", "AP50", "AP75",
                                     "AP_small", "AP_medium", "AP_large"]}
        coco_dt = coco_gt.loadRes(dt_json)
        e = COCOeval(coco_gt, coco_dt, "bbox")
        e.evaluate(); e.accumulate(); e.summarize()
    return {
        "mAP":       float(e.stats[0]),
        "AP50":      float(e.stats[1]),
        "AP75":      float(e.stats[2]),
        "AP_small":  float(e.stats[3]),
        "AP_medium": float(e.stats[4]),
        "AP_large":  float(e.stats[5]),
    }


torch.cuda.empty_cache()
model_r1.eval()
preds_path_r1 = WORK_DIR / "val_preds_retinanet_R1.json"
t0 = time.time()
predictions_to_coco_eval(val_index, model_r1, STRAT, SIZE, str(preds_path_r1),
                         score_thr=0.001, device=DEVICE)
print(f"inference: {(time.time()-t0)/60:.2f} min")

metrics_r1 = coco_eval_summary_safe(str(gt_fullval_ignore), str(preds_path_r1))
print("=" * 60)
print("R1 — full val:")
for k, v in metrics_r1.items():
    print(f"  {k:10s}: {v:.4f}")
save_metrics(metrics_r1, WORK_DIR / "metrics_retinanet_R1.json")
wandb.log({f"val/{k}": v for k, v in metrics_r1.items()})

pred letterbox-1280:   0%|          | 0/2846 [00:00<?, ?it/s]

wrote 944,829 predictions -> /kaggle/working/val_preds_retinanet_R1.json
inference: 17.78 min
R1 — full val:
  mAP       : 0.1082
  AP50      : 0.2592
  AP75      : 0.0734
  AP_small  : 0.0533
  AP_medium : 0.0998
  AP_large  : 0.1976


In [8]:
model_r1.eval()
if DEVICE == "cuda":
    torch.cuda.synchronize()

# warmup
for i in range(5):
    _ = predict_with_strategy_eval(model_r1, val_index.iloc[i]["image_path"],
                                    STRAT, SIZE, score_thr=0.5, device=DEVICE)

times = []
for i in range(50):
    row = val_index.iloc[(5 + i) % len(val_index)]
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    _ = predict_with_strategy_eval(model_r1, row["image_path"], STRAT, SIZE,
                                    score_thr=0.5, device=DEVICE)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    times.append(time.time() - t0)

mean_s = float(np.mean(times))
speed_r1 = {
    "device": str(DEVICE),
    "gpu_ms_per_image": mean_s * 1000,
    "gpu_fps":          1.0 / mean_s,
    "n_images":         50, "score_thr": 0.5,
}
print("speed:", json.dumps(speed_r1, indent=2))
with open(WORK_DIR / "speed_retinanet_R1.json", "w") as f:
    json.dump(speed_r1, f, indent=2)
wandb.log({"speed/gpu_fps": speed_r1["gpu_fps"],
           "speed/gpu_ms": speed_r1["gpu_ms_per_image"]})

speed: {
  "device": "cuda",
  "gpu_ms_per_image": 304.65017795562744,
  "gpu_fps": 3.2824533591628193,
  "n_images": 50,
  "score_thr": 0.5
}


In [9]:
test_df = pd.read_csv(TEST_CSV)
print(f"test frames: {len(test_df):,}")

submission_rows = []
model_r1.eval()
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="test inference"):
    fname = row["file_name"]
    img_path = str(COMP_DIR / fname) if not os.path.isabs(fname) else fname
    boxes, labels, scores, _, _ = predict_with_strategy_eval(
        model_r1, img_path, STRAT, SIZE, score_thr=0.05, device=DEVICE)

    if len(scores) > 0:
        order = np.argsort(-scores)[:500]
        boxes, labels, scores = boxes[order], labels[order], scores[order]

    parts = []
    for (x1, y1, x2, y2), l, s in zip(boxes, labels, scores):
        cls_id = int(l) - 1
        if cls_id < 0 or cls_id >= NUM_CLASSES:
            continue
        w, h = float(x2 - x1), float(y2 - y1)
        if w <= 0 or h <= 0:
            continue
        parts.append(f"{cls_id} {float(s):.4f} {x1:.1f} {y1:.1f} {w:.1f} {h:.1f}")

    submission_rows.append({
        "image_id": row["image_id"],
        "PredictionString": " ".join(parts),
    })

sub_df = pd.DataFrame(submission_rows)
sub_path = WORK_DIR / "submission_retinanet_R1.csv"
sub_df.to_csv(sub_path, index=False)
non_empty = (sub_df["PredictionString"].str.len() > 0).sum()
print(f"submission saved: {sub_path}")
print(f"non-empty: {non_empty}/{len(sub_df)} ({100*non_empty/len(sub_df):.1f}%)")

test frames: 3,811


test inference:   0%|          | 0/3811 [00:00<?, ?it/s]

submission saved: /kaggle/working/submission_retinanet_R1.csv
non-empty: 3811/3811 (100.0%)


In [10]:
print("=" * 70)
print("nb06 R1 COMPLETE — compare R0 vs R1")
print("=" * 70)
print()
print(f"{'Metric':<12} {'R0 (default)':<15} {'R1 (k-means)':<15} {'Δ':<10}")
r0_metrics = {
    "mAP": 0.0410, "AP50": 0.0941, "AP75": 0.0310,
    "AP_small": 0.0148, "AP_medium": 0.0352, "AP_large": 0.1053,
}
for k in ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]:
    r0 = r0_metrics[k]
    r1 = metrics_r1[k]
    print(f"{k:<12} {r0:<15.4f} {r1:<15.4f} {r1-r0:+.4f}")
print()
print(f"R0 GPU FPS: 3.39  | R1 GPU FPS: {speed_r1['gpu_fps']:.2f}")
print()
print("Artifacts in /kaggle/working/:")
for p in sorted(WORK_DIR.glob("*retinanet_R1*")):
    print(f"  {p.name}  ({p.stat().st_size/1e6:.1f} MB)")
print(f"  {sub_path.name}  ({sub_path.stat().st_size/1e6:.1f} MB)")
print()
print("Next: submit submission_retinanet_R1.csv to Kaggle LB")
print("=" * 70)

run.finish()
print("W&B run finished:", run.url)

wandb: updating run metadata


nb06 R1 COMPLETE — compare R0 vs R1

Metric       R0 (default)    R1 (k-means)    Δ         
mAP          0.0410          0.1082          +0.0672
AP50         0.0941          0.2592          +0.1651
AP75         0.0310          0.0734          +0.0424
AP_small     0.0148          0.0533          +0.0385
AP_medium    0.0352          0.0998          +0.0646
AP_large     0.1053          0.1976          +0.0923

R0 GPU FPS: 3.39  | R1 GPU FPS: 3.28

Artifacts in /kaggle/working/:
  metrics_retinanet_R1.json  (0.0 MB)
  retinanet_R1.pth  (146.3 MB)
  speed_retinanet_R1.json  (0.0 MB)
  submission_retinanet_R1.csv  (46.9 MB)
  train_log_retinanet_R1.json  (0.0 MB)
  val_preds_retinanet_R1.json  (142.8 MB)
  submission_retinanet_R1.csv  (46.9 MB)

Next: submit submission_retinanet_R1.csv to Kaggle LB


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:      speed/gpu_fps ▁
wandb:       speed/gpu_ms ▁
wandb:   train/epoch_loss ▁
wandb: train/epoch_time_h ▁
wandb:         train/loss ██▇▄▇▄▄▅▃▄▄▄▂▄▂▁▃▃▃▃▂▃▃▄▂▂▁▂▃▂▃▂▂▂▃▂▃▂▃▃
wandb:     train/loss_box █▆▅▄▄▅▅▃▅▃▅▄▄▅▃▄▁▃▄▄▂▄▃▃▃▃▂▂▃▃▁▂▃▄▃▂▄▂▂▄
wandb:     train/loss_cls █▆▆█▇▇▅▄▅▅▄▄▅▃▃▄▄▄▃▄▃▃▂▃▃▂▁▂▂▂▄▂▂▂▂▃▃▂▄▂
wandb:         train/step ▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇█
wandb:           val/AP50 ▁
wandb:           val/AP75 ▁
wandb:                 +4 ...
wandb: 
wandb: Run summary:
wandb:      speed/gpu_fps 3.28245
wandb:       speed/gpu_ms 304.65018
wandb:   train/epoch_loss 0.57648
wandb: train/epoch_time_h 4.86862
wandb:         train/loss 0.457
wandb:     train/loss_box 0.28645
wandb:     train/loss_cls 0.17055
wandb:         train/step 12080
wandb:           val/AP50 0.2592
wandb:           val/AP75 0.07341
wandb:                 +4 ...
wandb: 
wandb: 🚀 View run reti

W&B run finished: https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det/runs/nu184o9s
